In [20]:
import os
import glob
import numpy as np
import pretty_midi
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

def midi_to_pianoroll(file_path, fs=16):
    """Convert a MIDI file into a binary piano roll of shape [time, 128]."""
    midi = pretty_midi.PrettyMIDI(file_path)
    piano_roll = midi.get_piano_roll(fs=fs)  # [128, time]
    piano_roll = (piano_roll > 0).astype(np.float32)
    return piano_roll.T  # [time, 128]

def load_valid_pianorolls(file_list, fs=16):
    """Load MIDI files and skip unreadable/corrupt files safely."""
    pianorolls = []
    bad_files = []

    for fp in file_list:
        try:
            pr = midi_to_pianoroll(fp, fs=fs)
            if pr.shape[0] == 0:
                bad_files.append((fp, "empty piano roll"))
                continue
            pianorolls.append(pr)
        except Exception as exc:
            bad_files.append((fp, f"{type(exc).__name__}: {exc}"))

    return pianorolls, bad_files

def pianoroll_to_pretty_midi(piano_roll, fs=16, program=0, min_note_len=1):
    """Convert a binary piano roll [time, 128] back to a PrettyMIDI object."""
    pm = pretty_midi.PrettyMIDI()
    instrument = pretty_midi.Instrument(program=program)

    roll = (piano_roll > 0.5).astype(np.int32)
    time_per_step = 1.0 / fs

    for pitch in range(128):
        active = roll[:, pitch]
        padded = np.pad(active, (1, 1), mode="constant")
        changes = np.diff(padded)
        starts = np.where(changes == 1)[0]
        ends = np.where(changes == -1)[0]

        for s, e in zip(starts, ends):
            if (e - s) >= min_note_len:
                note = pretty_midi.Note(
                    velocity=90,
                    pitch=pitch,
                    start=s * time_per_step,
                    end=e * time_per_step,
                )
                instrument.notes.append(note)

    pm.instruments.append(instrument)
    return pm

class MIDIDataset(Dataset):
    def __init__(self, pieces, seq_len=64):
        self.pieces = pieces
        self.seq_len = seq_len
        self.indices = []

        for piece_idx, piece in enumerate(self.pieces):
            if piece.shape[0] > self.seq_len:
                max_start = piece.shape[0] - self.seq_len - 1
                for start in range(max_start + 1):
                    self.indices.append((piece_idx, start))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        piece_idx, start = self.indices[idx]
        piece = self.pieces[piece_idx]

        x_seq = piece[start : start + self.seq_len]
        y_seq = piece[start + 1 : start + self.seq_len + 1]

        return torch.from_numpy(x_seq).float(), torch.from_numpy(y_seq).float()

class MusicLSTM(nn.Module):
    def __init__(self, input_size=128, hidden_size=256, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.fc = nn.Linear(hidden_size, input_size)

    def forward(self, x, hidden=None):
        out, hidden = self.lstm(x, hidden)
        logits = self.fc(out)
        return logits, hidden

In [21]:
print(torch.cuda.is_available())

True


In [22]:
fpath = r"C:\Users\risha\Downloads\midi\data"
assert os.path.isdir(fpath), f"Folder not found: {fpath}"

In [23]:
files = sorted(glob.glob(os.path.join(fpath, "*.mid")))
len(files), files[:3]

(95,
 ['C:\\Users\\risha\\Downloads\\midi\\data\\Bizet Symphony in C 1mov.mid',
  'C:\\Users\\risha\\Downloads\\midi\\data\\Bizet Symphony in C 2mov.mid',
  'C:\\Users\\risha\\Downloads\\midi\\data\\Bizet Symphony in C 3mov.mid'])

In [24]:
if not files:
    raise ValueError("No MIDI files found in the folder.")

print(f"Found {len(files)} MIDI files")

Found 95 MIDI files


In [25]:
pianorolls, bad_files = load_valid_pianorolls(files, fs=16)
print(f"Loaded {len(pianorolls)} valid MIDI files")
print(f"Skipped {len(bad_files)} invalid MIDI files")
if bad_files:
    print("First skipped file:", bad_files[0][0])
    print("Reason:", bad_files[0][1])

if not pianorolls:
    raise ValueError("No valid MIDI files were loaded.")

len(pianorolls)

Loaded 93 valid MIDI files
Skipped 2 invalid MIDI files
First skipped file: C:\Users\risha\Downloads\midi\data\Buxethude Buxwv138 Prelude.mid
Reason: IndexError: list index out of range


93

In [26]:
first = pianorolls[0]
first.shape, float(first.max()), float(first.min())

((10245, 128), 1.0, 0.0)

In [27]:
lengths = [p.shape[0] for p in pianorolls]
print(f"Min length: {min(lengths)}")
print(f"Max length: {max(lengths)}")
print(f"Mean length: {np.mean(lengths):.1f}")

Min length: 115
Max length: 16463
Mean length: 4897.9


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
seq_len = 64
batch_size = 64

train_dataset = MIDIDataset(pianorolls, seq_len=seq_len)
if len(train_dataset) == 0:
    raise ValueError("Dataset has no training sequences. Reduce seq_len or check MIDI files.")

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    pin_memory=(device.type == "cuda"),
)

print(f"Device: {device}")
print(f"Training sequences: {len(train_dataset)}")

Device: cuda
Training sequences: 449555


In [29]:
model = MusicLSTM(input_size=128, hidden_size=256, num_layers=2, dropout=0.3).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 30
grad_clip = 1.0
max_batches_per_epoch = None  # Set to None for full epoch training

In [ ]:
model.train()
for epoch in range(1, epochs + 1):
    total_loss = 0.0
    num_batches = 0

    for x_batch, y_batch in train_loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        logits, _ = model(x_batch)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1

        if max_batches_per_epoch is not None and num_batches >= max_batches_per_epoch:
            break

    avg_loss = total_loss / max(1, num_batches)
    print(f"Epoch {epoch:02d}/{epochs} | Loss: {avg_loss:.4f} | Batches: {num_batches}")

In [55]:
@torch.no_grad()
def generate_music(model, seed_seq, steps=256, temperature=1.0):
    model.eval()
    generated = seed_seq.clone().to(device)  # [seq_len, 128]

    hidden = None
    inp = generated.unsqueeze(0)  # [1, seq_len, 128]

    for _ in range(steps):
        logits, hidden = model(inp, hidden)
        next_logits = logits[:, -1, :] / max(temperature, 1e-6)
        probs = torch.sigmoid(next_logits)
        next_step = torch.bernoulli(probs)  # [1, 128]

        generated = torch.cat([generated, next_step], dim=0)
        inp = next_step.unsqueeze(1)  # [1, 1, 128]

    return generated.cpu().numpy()

seed_piece = pianorolls[0]
seed = torch.from_numpy(seed_piece[:seq_len]).float()
generated_roll = generate_music(model, seed, steps=512, temperature=1.0)

out_pm = pianoroll_to_pretty_midi(generated_roll, fs=16, program=0)
out_path = os.path.join("C:\\Users\\risha\\Downloads\\midi\\", "out.mid")
out_pm.write(out_path)
out_path

'C:\\Users\\risha\\Downloads\\midi\\out.mid'

In [53]:
# Optional: save trained weights
model_path = os.path.join(fpath, "music_lstm.pt")
torch.save(model.state_dict(), model_path)
model_path

'C:\\Users\\risha\\Downloads\\midi\\data\\music_lstm.pt'